# SuperNEMO Transformer token-count audit

This notebook measures the **real, unpadded token counts** for the `2nubb` versus `Bi214` benchmark before we freeze a tokenizer. It does not train a model and it does not import the existing SuperNEMO tokenizer.

The three representation families match the other detector benchmarks:

1. **Entity tokens:** one tracker hit is one token. Token coordinates are the hit's `(tX, tY, tZ)` position. Proposed content is normalized `tR`, a valid-`tR` indicator, and event hit-count context.
2. **Patch tokens:** hits in the same fixed detector-space region are aggregated into one token. The token coordinate is the region's hit centroid; content can include occupancy, mean/max `tR`, valid-radius fraction, and spatial spread. This notebook tests several candidate spatial bin sizes.
3. **Summary-feature tokens:** hits are put in a deterministic locality-preserving order and divided into balanced groups. Each group becomes one statistical summary token. This notebook measures several hits-per-summary-group choices.

`E1`, `E2`, reconstructed angles, labels, and source names are not model inputs. `E1 + E2` remains evaluation-only. A shared capacity of 512 does **not** imply that every event contains 512 real tokens.

In [ ]:
from pathlib import Path
import json
import math
import time

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Lab-computer defaults. Edit only if the project or data live elsewhere.
DATA_ROOT = Path('/home/klz/Data/zeronu_benchmark/SuperNEMO')
WING_MANIFEST_DIR = Path('/home/wenyu/SuperNEMO/data/manifests')
OUTPUT_DIR = Path('/home/klz/Data/zeronu_benchmark/Transformer_Approach/supernemo_detector/analysis/token_count_audit')

FILES = {
    '2nu': DATA_ROOT / 'data_2nubb_merged.h5',
    'Bi214': DATA_ROOT / 'data_Bi214_merged.h5',
}
OFFSET_NAMES = {'2nu': '2nubb_event_offsets.npy', 'Bi214': 'Bi214_event_offsets.npy'}

MAX_TOKENS = 512
SAMPLE_EVENTS_PER_CLASS = 20_000  # Increase after the first successful run if desired.
RANDOM_SEED = 42
SCAN_CHUNK_ROWS = 2_000_000
PATCH_BIN_SIZES_MM = (44.0, 60.0, 88.0)
SUMMARY_HITS_PER_GROUP = (2, 4, 8)
BATCH_SIZE = 64

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for category, path in FILES.items():
    if not path.is_file():
        raise FileNotFoundError(f'{category}: missing {path}')
print('Data files found.')
print('Outputs:', OUTPUT_DIR)

## Complete-event indexes

Each HDF5 row is one tracker hit, while consecutive rows with the same `ev_no` form one event. Wing's offset arrays give the row boundaries of every complete event. The notebook reuses those arrays when present; otherwise it builds audit-only indexes by streaming `ev_no` and saves them under the output directory.

In [ ]:
def validate_offsets(offsets, row_count, context):
    offsets = np.asarray(offsets)
    if offsets.ndim != 1 or offsets.dtype != np.int64:
        raise ValueError(f'{context}: offsets must be a one-dimensional int64 array')
    if len(offsets) < 2 or offsets[0] != 0 or offsets[-1] != row_count:
        raise ValueError(f'{context}: offset endpoints do not match the HDF5 rows')
    if np.any(np.diff(offsets) <= 0):
        raise ValueError(f'{context}: every event must contain at least one row')
    return offsets


def build_offsets(h5_path, output_path, chunk_rows=SCAN_CHUNK_ROWS):
    print(f'Building event offsets for {h5_path.name} ...')
    started = time.time()
    boundaries = [np.asarray([0], dtype=np.int64)]
    with h5py.File(h5_path, 'r') as handle:
        event_ids = handle['ev_no']
        row_count = len(event_ids)
        previous = None
        for start in range(0, row_count, chunk_rows):
            stop = min(start + chunk_rows, row_count)
            values = np.asarray(event_ids[start:stop], dtype=np.int64)
            if values.size == 0:
                continue
            if previous is not None:
                step = int(values[0]) - previous
                if step not in (0, 1):
                    raise ValueError(f'ev_no gap/decrease at row {start}')
                if step == 1:
                    boundaries.append(np.asarray([start], dtype=np.int64))
            differences = np.diff(values)
            if np.any((differences < 0) | (differences > 1)):
                bad = start + int(np.flatnonzero((differences < 0) | (differences > 1))[0]) + 1
                raise ValueError(f'ev_no gap/decrease at row {bad}')
            changes = np.flatnonzero(differences == 1).astype(np.int64) + start + 1
            if len(changes):
                boundaries.append(changes)
            previous = int(values[-1])
        boundaries.append(np.asarray([row_count], dtype=np.int64))
    offsets = np.concatenate(boundaries).astype(np.int64, copy=False)
    offsets = validate_offsets(offsets, row_count, h5_path.name)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(output_path, offsets, allow_pickle=False)
    print(f'Built {len(offsets)-1:,} events in {(time.time()-started)/60:.2f} min -> {output_path}')
    return offsets


def load_or_build_offsets(category, h5_path):
    with h5py.File(h5_path, 'r') as handle:
        row_count = len(handle['ev_no'])
    candidates = [
        WING_MANIFEST_DIR / OFFSET_NAMES[category],
        OUTPUT_DIR / 'offsets' / OFFSET_NAMES[category],
    ]
    for candidate in candidates:
        if candidate.is_file():
            offsets = np.load(candidate, mmap_mode='r', allow_pickle=False)
            validate_offsets(offsets, row_count, str(candidate))
            print(f'{category}: using {candidate}')
            return offsets
    return build_offsets(h5_path, candidates[-1])


offsets_by_category = {
    category: load_or_build_offsets(category, path)
    for category, path in FILES.items()
}


In [ ]:
# Exact hit-count distributions use the complete offset arrays and do not sample events.
exact_rows = []
for category, offsets in offsets_by_category.items():
    hit_counts = np.diff(offsets)
    quantiles = np.quantile(hit_counts, [0, .5, .9, .95, .99, .999, 1.0])
    exact_rows.append({
        'category': category,
        'events': len(hit_counts),
        'mean_hits': float(hit_counts.mean()),
        'min': int(quantiles[0]),
        'p50': float(quantiles[1]),
        'p90': float(quantiles[2]),
        'p95': float(quantiles[3]),
        'p99': float(quantiles[4]),
        'p99.9': float(quantiles[5]),
        'max': int(quantiles[6]),
        'fraction_over_512': float(np.mean(hit_counts > MAX_TOKENS)),
    })
exact_hit_summary = pd.DataFrame(exact_rows)
display(exact_hit_summary)
exact_hit_summary.to_csv(OUTPUT_DIR / 'exact_hit_count_summary.csv', index=False)

## Token-count definitions used in this audit

- **Entity:** `min(number of hits, 512)`. Coverage is the retained-hit fraction.
- **Patch:** assign raw detector-space XYZ positions to fixed cubic bins, count occupied bins, and retain at most 512. If more than 512 bins are occupied, retain the most populated bins and report their hit coverage. Centering for positional encoding happens only after grouping.
- **Summary:** report balanced groups containing approximately 2, 4, or 8 hits each. All hits contribute, so coverage is one. The final implementation would use a deterministic spatial ordering such as a Morton code before forming these groups.

Token count alone does not choose the best representation. This audit tells us whether a proposed cap or grouping rule actually changes the data.

In [ ]:
def patch_count_and_coverage(xyz, bin_size_mm, max_tokens=MAX_TOKENS):
    # Fixed detector-space bins: do not center XYZ before this operation.
    cells = np.floor(np.asarray(xyz, dtype=np.float64) / float(bin_size_mm)).astype(np.int64)
    _, occupancy = np.unique(cells, axis=0, return_counts=True)
    if len(occupancy) <= max_tokens:
        return len(occupancy), 1.0
    retained = np.sort(occupancy)[-max_tokens:].sum(dtype=np.int64)
    return max_tokens, float(retained / len(xyz))


def nearest_nonzero_distance(xyz):
    xyz = np.asarray(xyz, dtype=np.float64)
    if len(xyz) < 2:
        return np.nan
    delta = xyz[:, None, :] - xyz[None, :, :]
    distances = np.sqrt(np.sum(delta * delta, axis=-1))
    distances[distances == 0.0] = np.inf
    result = np.min(distances, axis=1)
    finite = result[np.isfinite(result)]
    return float(np.median(finite)) if len(finite) else np.nan


def audit_sample(category, h5_path, offsets, sample_size, seed):
    event_count = len(offsets) - 1
    sample_size = min(int(sample_size), event_count)
    category_seed = seed + (0 if category == '2nu' else 1)
    generator = np.random.default_rng(category_seed)
    selected = np.sort(generator.choice(event_count, size=sample_size, replace=False))
    rows = []
    started = time.time()
    with h5py.File(h5_path, 'r') as handle:
        x, y, z, radius = handle['tX'], handle['tY'], handle['tZ'], handle['tR']
        for position, event_index in enumerate(selected, start=1):
            start, stop = int(offsets[event_index]), int(offsets[event_index + 1])
            xyz = np.column_stack((x[start:stop], y[start:stop], z[start:stop])).astype(np.float32)
            tr = np.asarray(radius[start:stop], dtype=np.float32)
            number_of_hits = len(xyz)
            row = {
                'category': category,
                'event_index': int(event_index),
                'hits': number_of_hits,
                'missing_tR_fraction': float(np.mean(~np.isfinite(tr))),
                'entity_tokens': min(number_of_hits, MAX_TOKENS),
                'entity_coverage': min(number_of_hits, MAX_TOKENS) / number_of_hits,
                # Nearest-neighbor geometry is sampled more narrowly because it is quadratic.
                'median_nearest_hit_distance_mm': nearest_nonzero_distance(xyz) if position <= 2000 else np.nan,
            }
            for bin_size in PATCH_BIN_SIZES_MM:
                count, coverage = patch_count_and_coverage(xyz, bin_size)
                tag = str(bin_size).replace('.', 'p')
                row[f'patch_{tag}mm_tokens'] = count
                row[f'patch_{tag}mm_coverage'] = coverage
            for group_size in SUMMARY_HITS_PER_GROUP:
                row[f'summary_{group_size}hits_tokens'] = min(
                    math.ceil(number_of_hits / group_size), MAX_TOKENS
                )
                row[f'summary_{group_size}hits_coverage'] = 1.0
            rows.append(row)
            if position % 5000 == 0:
                print(f'{category}: {position:,}/{sample_size:,} events')
    print(f'{category}: completed in {(time.time()-started)/60:.2f} min')
    return pd.DataFrame(rows)


sample_frames = [
    audit_sample(category, FILES[category], offsets, SAMPLE_EVENTS_PER_CLASS, RANDOM_SEED)
    for category, offsets in offsets_by_category.items()
]
audit = pd.concat(sample_frames, ignore_index=True)
audit.to_csv(OUTPUT_DIR / 'sampled_event_token_counts.csv', index=False)
audit.head()

In [ ]:
token_columns = [column for column in audit if column.endswith('_tokens')]
coverage_columns = [column for column in audit if column.endswith('_coverage')]

def summarize_columns(frame, columns):
    records = []
    for category, group in frame.groupby('category', sort=False):
        for column in columns:
            values = group[column].to_numpy(dtype=np.float64)
            records.append({
                'category': category,
                'representation': column,
                'mean': float(values.mean()),
                'p50': float(np.quantile(values, .50)),
                'p90': float(np.quantile(values, .90)),
                'p95': float(np.quantile(values, .95)),
                'p99': float(np.quantile(values, .99)),
                'max': float(values.max()),
                'fraction_at_512': float(np.mean(values >= MAX_TOKENS)),
            })
    return pd.DataFrame(records)

token_summary = summarize_columns(audit, token_columns)
coverage_summary = summarize_columns(audit, coverage_columns)
token_summary.to_csv(OUTPUT_DIR / 'sampled_token_count_summary.csv', index=False)
coverage_summary.to_csv(OUTPUT_DIR / 'sampled_coverage_summary.csv', index=False)

display(token_summary)
display(coverage_summary)
print('Sample missing-tR fraction:', audit['missing_tR_fraction'].mean())
print('Sample median nearest-hit distance (mm):', audit['median_nearest_hit_distance_mm'].median())

In [ ]:
# Estimate executed sequence length under dynamic batch trimming.
# This uses the sampled event pool and should be treated as a systems estimate, not a metric.
def batch_padding_summary(values, batch_size=BATCH_SIZE, seed=RANDOM_SEED):
    values = np.asarray(values, dtype=np.int64).copy()
    np.random.default_rng(seed).shuffle(values)
    maxima = [int(values[start:start+batch_size].max()) for start in range(0, len(values), batch_size)]
    maxima = np.asarray(maxima, dtype=np.float64)
    return {
        'mean_valid_tokens_per_event': float(values.mean()),
        'mean_executed_batch_length': float(maxima.mean()),
        'max_executed_batch_length': int(maxima.max()),
        'attention_position_fraction_vs_512': float(np.mean(maxima ** 2) / MAX_TOKENS ** 2),
    }

batch_rows = []
for column in token_columns:
    batch_rows.append({'representation': column, **batch_padding_summary(audit[column])})
batch_summary = pd.DataFrame(batch_rows)
display(batch_summary)
batch_summary.to_csv(OUTPUT_DIR / 'dynamic_batch_length_estimates.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for category, group in audit.groupby('category', sort=False):
    axes[0].hist(group['hits'], bins=60, histtype='step', linewidth=1.8, label=category)
axes[0].axvline(MAX_TOKENS, color='black', linestyle='--', label='512-token cap')
axes[0].set_title('Tracker hits per sampled event')
axes[0].set_xlabel('Hits')
axes[0].set_ylabel('Sampled events')
axes[0].set_yscale('log')
axes[0].legend()

plot_columns = ['entity_tokens'] + [c for c in token_columns if c.startswith('patch_')] + [c for c in token_columns if c.startswith('summary_')]
means = audit[plot_columns].mean().sort_values(ascending=False)
axes[1].barh(means.index, means.values)
axes[1].invert_yaxis()
axes[1].set_title('Mean real tokens in the sampled pool')
axes[1].set_xlabel('Mean valid tokens per event')

fig.tight_layout()
figure_path = OUTPUT_DIR / 'token_count_audit.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', figure_path)

## How to interpret the results

1. If essentially no entity sequence reaches 512, keep 512 as the shared capacity but describe the representation using its actual token distribution. Do not duplicate hits.
2. Prefer a patch size near the verified tracker-cell spacing that changes the representation without collapsing most events to only one or two tokens. Check both token count and retained-hit coverage.
3. Choose a summary group size that creates a meaningful compression relative to entity tokens. If two-hit groups barely change sequence length, compare four- or eight-hit groups.
4. Check results separately for `2nu` and `Bi214`. A tokenizer should not truncate one class much more aggressively than the other.
5. Use only the training split for the final parameter choice once Wing's manifest is loaded. This notebook samples the complete source files for a representation audit and must not use token statistics to optimize classification performance on the test set.
6. After freezing the settings, record the tokenizer configuration and source hash with every run.